<a href="https://colab.research.google.com/github/Ahmed-Saeed-Abdullah-Alshanwany/Al_Game_Character_Designer/blob/main/Anime_Game_Character_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🎮 Anime Game Character Designer - Fine-Tuning Edition

SD 1.5 + LORA | Dataset: HuggingFace (Anime)

Optimized for Google Colab T4 (15 GB VRAM)

---



In [ ]:
!pip uninstall -y websockets fsspec

Found existing installation: websockets 15.0.1
Uninstalling websockets-15.0.1:
  Successfully uninstalled websockets-15.0.1
Found existing installation: fsspec 2025.3.0
Uninstalling fsspec-2025.3.0:
  Successfully uninstalled fsspec-2025.3.0


In [ ]:
# ───────────────────────────────────────────────
# CELL 1 — Install Dependencies
# ───────────────────────────────────────────────
!pip install -q \
    "diffusers==0.30.3" \
    "huggingface_hub==0.23.4" \
    "transformers==4.41.2" \
    accelerate \
    safetensors \
    peft \
    datasets \
    gradio \
    Pillow \
    torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 136.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not cu

In [ ]:
# Restart the runtime after installation
import os
os.kill(os.getpid(), 9)

In [ ]:
# ───────────────────────────────────────────────
# CELL 2 — Imports & GPU Check
# ───────────────────────────────────────────────
import os, math, gc, torch
from pathlib import Path
from PIL import Image
import gradio as gr

from diffusers import (
    StableDiffusionPipeline,
    DDPMScheduler,
    AutoencoderKL,
    UNet2DConditionModel,
)
from diffusers.optimization import get_scheduler
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from accelerate import Accelerator
from datasets import load_dataset

assert torch.cuda.is_available(), "CUDA is not available !"
print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

✅ GPU : Tesla T4
✅ VRAM: 15.6 GB


In [ ]:
# ───────────────────────────────────────────────
# CELL 3 — Configuration
# ───────────────────────────────────────────────
CFG = dict(
    model_id         = "runwayml/stable-diffusion-v1-5",
    output_dir       = "/content/lora_anime",

    # -- Hugging Face Dataset ------------------
    hf_dataset       = "huggan/anime-faces",
    hf_image_col     = "image",      # column name containing images
    max_samples      = 200,          # number of images to use for training

    # -- Prompts --------------------------------
    instance_prompt  = "anime character, sks style",
    neg_prompt       = "blurry, low quality, deformed, watermark, text",

    # -- Training Hyperparameters ---------------
    resolution       = 512,
    train_batch      = 1,
    grad_accum       = 4,            # effective batch size = 4
    lr               = 1e-4,
    num_epochs       = 40,
    save_every       = 10,           # save checkpoint every N epochs
    lora_rank        = 8,
    mixed_precision  = "fp16",       # required for T4
    seed             = 42,
)
Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)

In [ ]:
from datasets import load_dataset
import itertools

print("Downloading anime dataset (streaming mode) ...")

ds = load_dataset(
    CFG["hf_dataset"],
    split="train",
    streaming=True
)

ds = ds.shuffle(
    seed=CFG["seed"],
    buffer_size=1000
)

ds = itertools.islice(ds, CFG["max_samples"])
ds = list(ds)
print(f"Dataset loaded: {len(ds)} images")
img_col = next(
    (c for c in ds[0].keys() if "image" in c.lower() or "img" in c.lower()),
    list(ds[0].keys())[0]
)

print(f"Image column detected: '{img_col}'")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/43103 [00:00<?, ?it/s]

Dataset loaded: 200 images
Image column detected: 'image'


In [ ]:
# ───────────────────────────────────────────────
# CELL 5 — Dataset Class
# Reads directly from a Hugging Face dataset object
# ───────────────────────────────────────────────
class AnimeHFDataset(Dataset):
    def __init__(self, hf_ds, tokenizer, prompt, img_col, size=512):
        self.ds        = hf_ds
        self.tokenizer = tokenizer
        self.prompt    = prompt
        self.img_col   = img_col
        self.tfm = transforms.Compose([
            transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(size),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        item = self.ds[i]
        img  = item[self.img_col]

        # HF datasets usually return PIL Images directly
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        img = img.convert("RGB")

        pixel = self.tfm(img)
        ids   = self.tokenizer(
            self.prompt,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        return {"pixel_values": pixel, "input_ids": ids}

In [ ]:
# ───────────────────────────────────────────────
# CELL 6 — Load Model Components
# VAE and Text Encoder are frozen — only UNet is trained
# ───────────────────────────────────────────────
print("Loading SD 1.5 components ...")
tokenizer   = CLIPTokenizer.from_pretrained(CFG["model_id"], subfolder="tokenizer")
text_enc    = CLIPTextModel.from_pretrained(CFG["model_id"], subfolder="text_encoder")
vae         = AutoencoderKL.from_pretrained(CFG["model_id"], subfolder="vae")
unet        = UNet2DConditionModel.from_pretrained(CFG["model_id"], subfolder="unet")
noise_sched = DDPMScheduler.from_pretrained(CFG["model_id"], subfolder="scheduler")

# Freeze VAE and text encoder
vae.requires_grad_(False)
text_enc.requires_grad_(False)
print("Model loaded successfully")

Loading SD 1.5 components ...


tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

...

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
# ───────────────────────────────────────────────
# CELL 7 — Apply LoRA to UNet
# ───────────────────────────────────────────────
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r             = CFG["lora_rank"],
    lora_alpha    = CFG["lora_rank"],
    target_modules= ["to_q", "to_v", "to_k", "to_out.0"],
    lora_dropout  = 0.05,
    bias          = "none",
)
unet = get_peft_model(unet, lora_cfg)
unet.print_trainable_parameters()

trainable params: 1,594,368 || all params: 861,115,332 || trainable%: 0.1852


In [ ]:
# ───────────────────────────────────────────────
# CELL 8 — Training Loop
# ───────────────────────────────────────────────
import math
from diffusers.optimization import get_scheduler as get_lr_scheduler
from accelerate import Accelerator

torch.manual_seed(CFG["seed"])

dataset    = AnimeHFDataset(ds, tokenizer, CFG["instance_prompt"], img_col, CFG["resolution"])
dataloader = DataLoader(dataset, batch_size=CFG["train_batch"], shuffle=True, num_workers=2)

optimizer  = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=CFG["lr"], weight_decay=1e-2
)
total_steps  = math.ceil(len(dataloader) / CFG["grad_accum"]) * CFG["num_epochs"]
lr_scheduler = get_lr_scheduler(
    "cosine", optimizer=optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)

accelerator = Accelerator(
    mixed_precision=CFG["mixed_precision"],
    gradient_accumulation_steps=CFG["grad_accum"],
)
unet, text_enc, vae, optimizer, dataloader, lr_scheduler = accelerator.prepare(
    unet, text_enc, vae, optimizer, dataloader, lr_scheduler
)

dtype = torch.float16
print(f"\nTraining started | {CFG['num_epochs']} epochs | {total_steps} total steps\n")

for epoch in range(CFG["num_epochs"]):
    unet.train()
    epoch_loss = 0.0

    for batch in dataloader:
        with accelerator.accumulate(unet):
            latents    = vae.encode(batch["pixel_values"].float()).latent_dist.sample().to(dtype) * 0.18215
            noise      = torch.randn_like(latents)
            timesteps  = torch.randint(0, noise_sched.config.num_train_timesteps,
                                       (latents.shape[0],), device=latents.device).long()
            noisy_lat  = noise_sched.add_noise(latents, noise, timesteps)
            enc_hidden = text_enc(batch["input_ids"])[0]
            pred       = unet(noisy_lat, timesteps, enc_hidden).sample
            loss       = torch.nn.functional.mse_loss(pred.float(), noise.float())
            accelerator.backward(loss)

            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(unet.parameters(), 1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        epoch_loss += loss.detach().item()

    avg = epoch_loss / len(dataloader)
    print(f"Epoch [{epoch+1:3d}/{CFG['num_epochs']}]  loss: {avg:.4f}  lr: {lr_scheduler.get_last_lr()[0]:.2e}")

    if (epoch + 1) % CFG["save_every"] == 0:
        ckpt = f"{CFG['output_dir']}/epoch_{epoch+1}"
        accelerator.unwrap_model(unet).save_pretrained(ckpt)
        print(f"  Checkpoint saved -> {ckpt}")

accelerator.unwrap_model(unet).save_pretrained(f"{CFG['output_dir']}/final")
print("\n✅ Fine-tuning complete!")


Training started | 40 epochs | 2000 total steps

Epoch [  1/40]  loss: 0.1181  lr: 2.50e-05
Epoch [  2/40]  loss: 0.1064  lr: 5.00e-05
Epoch [  3/40]  loss: 0.1024  lr: 7.50e-05
Epoch [  4/40]  loss: 0.0903  lr: 1.00e-04
Epoch [  5/40]  loss: 0.1060  lr: 9.98e-05
Epoch [  6/40]  loss: 0.1088  lr: 9.92e-05
Epoch [  7/40]  loss: 0.1013  lr: 9.83e-05
Epoch [  8/40]  loss: 0.0790  lr: 9.70e-05
Epoch [  9/40]  loss: 0.0998  lr: 9.53e-05
Epoch [ 10/40]  loss: 0.0865  lr: 9.33e-05
  Checkpoint saved -> /content/lora_anime/epoch_10
Epoch [ 11/40]  loss: 0.0825  lr: 9.10e-05
Epoch [ 12/40]  loss: 0.0935  lr: 8.83e-05
Epoch [ 13/40]  loss: 0.0893  lr: 8.54e-05
Epoch [ 14/40]  loss: 0.0973  lr: 8.21e-05
Epoch [ 15/40]  loss: 0.0988  lr: 7.87e-05
Epoch [ 16/40]  loss: 0.0885  lr: 7.50e-05
Epoch [ 17/40]  loss: 0.0961  lr: 7.11e-05
Epoch [ 18/40]  loss: 0.0880  lr: 6.71e-05
Epoch [ 19/40]  loss: 0.0956  lr: 6.29e-05
Epoch [ 20/40]  loss: 0.0899  lr: 5.87e-05
  Checkpoint saved -> /content/lora_ani

In [ ]:
# ───────────────────────────────────────────────
# CELL 9 — Load Fine-Tuned Pipeline
# ───────────────────────────────────────────────
import gc
from peft import PeftModel
from diffusers import StableDiffusionPipeline

del optimizer, dataloader, lr_scheduler, accelerator
gc.collect()
torch.cuda.empty_cache()

base_unet = UNet2DConditionModel.from_pretrained(
    CFG["model_id"], subfolder="unet", torch_dtype=torch.float16
)
ft_unet = PeftModel.from_pretrained(base_unet, f"{CFG['output_dir']}/final")
ft_unet = ft_unet.merge_and_unload()

pipe = StableDiffusionPipeline.from_pretrained(
    CFG["model_id"], unet=ft_unet, torch_dtype=torch.float16
).to("cuda")
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
print("✅ Fine-tuned pipeline is ready!")

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Fine-tuned pipeline is ready!


In [ ]:
import torch
import gradio as gr
import gradio_client.utils as gr_utils

# --- Patch ---
def patched_get_type(schema):
    if isinstance(schema, bool):
        return "boolean"
    if "const" in schema:
        return "const"
    if "enum" in schema:
        return "enum"
    elif "type" in schema:
        return schema["type"]
    elif schema.get("$ref"):
        return "$ref"
    elif schema.get("oneOf"):
        return "oneOf"
    elif schema.get("anyOf"):
        return "anyOf"
    elif schema.get("allOf"):
        return "allOf"
    elif "type" not in schema:
        return {}
    else:
        raise ValueError(f"Cannot parse type for {schema}")

gr_utils.get_type = patched_get_type

EXAMPLES = [
    ["warrior with red armor and katana",    "blurry, low quality", 7.5, 30, 42],
    ["mage girl with purple magic aura",     "blurry, low quality", 7.5, 30, 0],
    ["cyberpunk ninja with neon highlights", "blurry, low quality", 8.0, 35, 7],
]

device = "cuda" if torch.cuda.is_available() else "cpu"

def generate(prompt, neg, guidance, steps, seed):
    full_prompt = f"anime character, sks style, {prompt}, high quality, detailed"
    generator = None if int(seed) == 0 else torch.Generator(device=device).manual_seed(int(seed))

    result = pipe(
        prompt=full_prompt,
        negative_prompt=neg,
        guidance_scale=float(guidance),
        num_inference_steps=int(steps),
        generator=generator,
    )
    return result.images[0]

ui = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="Character Description", value="warrior with golden armor"),
        gr.Textbox(label="Negative Prompt", value="blurry, low quality, deformed"),
        gr.Slider(1, 15, value=7.5, step=0.5, label="Guidance Scale"),
        gr.Slider(10, 50, value=30, step=5, label="Inference Steps"),
        gr.Number(value=0, label="Seed (0 = random)"),
    ],
    outputs=gr.Image(label="Generated Anime Character"),
    examples=EXAMPLES,
    title="Anime Game Character Designer — Fine-Tuned",
    description="Stable Diffusion 1.5 fine-tuned with LoRA on anime characters.\nTrigger word: anime character, sks style",
)

ui.launch(share=True, debug=True, inline=False)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://518e13b04d0bba7ac6.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]